# ByT5 LoRA Fine-tuning for Akkadian Translation

This notebook trains a LoRA adapter on google/byt5-xl for Akkadian transliteration → English translation.

**Pipeline:**
1. Load train/validation splits from `data/split_data/`
2. Load blind test data from `data/clean_data/test_clean.csv`
3. Tokenize and preprocess
4. Load model + apply quantization + LoRA
5. Train
6. Validate and print metrics
7. Save model
8. Generate predictions on test data
9. Save predictions to CSV

In [ ]:
import os
import json
import csv
from pathlib import Path
import pandas as pd
import torch
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"PyTorch version: {torch.__version__}")

# Prefer MPS (Apple Silicon) first, then CUDA, then CPU
DEVICE = 'cpu'
mps_available = False
try:
    mps_available = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available() and torch.backends.mps.is_built()
except Exception:
    mps_available = False

if mps_available:
    DEVICE = 'mps'
    print("✓ MPS (Apple Silicon) available and selected")
elif torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f"✓ CUDA GPU available: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = 'cpu'
    print("⚠ Using CPU (no MPS or CUDA available)")

print(f"Device: {DEVICE}\n")

PyTorch version: 2.10.0
✓ MPS (Apple Silicon) available and selected
Device: mps



## Configuration

In [2]:
# Hyperparameters & paths
CONFIG = {
    'model_name': 'google/byt5-base', # byt5-xl is too large for my macbook
    'train_file': 'data/split_data/train.jsonl',
    'validation_file': 'data/split_data/validation.jsonl',
    'test_file': 'data/clean_data/test_clean.csv',
    'output_dir': 'byt5_lora_ckpt',
    'predictions_file': 'byt5_lora_ckpt/test_predictions.csv',
    
    # Training
    'num_train_epochs': 3,
    'gradient_accumulation_steps': 64, # less frequent gradient updates increases effective batch size (speeds up training)
    'per_device_train_batch_size': 1, # batch size controls memory usage (we are very limited)
    'per_device_eval_batch_size': 1,
    'learning_rate': 1e-4,
    'warmup_steps': 100,
    'weight_decay': 0.01,
    'logging_steps': 100,
    'eval_steps': 500,
    'save_steps': 500,
    'save_total_limit': 3,
    
    # Tokenizer
    'max_input_length': 512,
    'max_target_length': 512,
    
    # Quantization: 0 = no quant, 4 = 4-bit, 8 = 8-bit
    'quant_bits': 0, # bitsandbytes does not work on apple silicon, disable quantization
    
    # LoRA
    'lora_r': 16,
    'lora_alpha': 32,
    'lora_dropout': 0.05,
    'lora_targets': ["q", "v"], # "k" and "o" are too much for my macbook to handle
}

# Convert paths to absolute
notebook_dir = Path('.').resolve()
for key in ['train_file', 'validation_file', 'test_file', 'output_dir', 'predictions_file']:
    path = notebook_dir / CONFIG[key]
    CONFIG[key] = str(path)
    print(f"{key}: {CONFIG[key]}")

print(f"\nConfig: {json.dumps(CONFIG, indent=2, default=str)}")

train_file: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/split_data/train.jsonl
validation_file: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/split_data/validation.jsonl
test_file: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/clean_data/test_clean.csv
output_dir: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt
predictions_file: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt/test_predictions.csv

Config: {
  "model_name": "google/byt5-base",
  "train_file": "/Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/split_data/train.jsonl",
  "validation_file": "/Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/split_data/validation.jsonl",
  "test_file": "/Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/data/clean_data/test_clean.csv",
  "output_dir": "/Users/adi

## Load Train/Validation Datasets

In [3]:
# Load from JSONL
data_files = {
    'train': CONFIG['train_file'],
    'validation': CONFIG['validation_file']
}

ds = load_dataset('json', data_files=data_files)
print(f"Train dataset: {len(ds['train'])} samples")
print(f"Validation dataset: {len(ds['validation'])} samples")

# Show sample
sample = ds['train'][0]
print(f"\nSample input length: {len(sample['input_text'])} chars")
print(f"Sample target length: {len(sample['target_text'])} chars")
print(f"Input (first 200 chars): {sample['input_text'][:200]}...")
print(f"Target (first 200 chars): {sample['target_text'][:200]}...")

Train dataset: 2269 samples
Validation dataset: 252 samples

Sample input length: 801 chars
Sample target length: 271 chars
Input (first 200 chars): um-ma wa-šu-na-lam₅-ma a-na ša-lim-a-šur qí-bi-ma {to say, speak, command} i-nu-mì {when; in those days; then, since then} a-na-kam {here, to/as far as/from here} tù-uš-bu {to sit (down); dwell; to ad...
Target (first 200 chars): From Wašunalam to Šalim-Aššur: When you stayed here you gave me your promise about some iron, and concerning that which I asked you for, you said: While I am staying here I shall write so they may bri...


## Initialize Tokenizer & Preprocessing

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'], use_fast=True)
print(f"Tokenizer loaded: {CONFIG['model_name']}")
print(f"Vocab size: {tokenizer.vocab_size}")

def preprocess_function(batch):
    """Tokenize input and target texts for seq2seq."""
    # Encode inputs (encoder)
    inputs = tokenizer(
        batch['input_text'],
        truncation=True,
        padding='max_length',
        max_length=CONFIG['max_input_length']
    )
    
    # Encode targets (decoder labels)
    targets = tokenizer(
        batch['target_text'],
        truncation=True,
        padding='max_length',
        max_length=CONFIG['max_target_length']
    )
    
    # T5/ByT5 automatically creates decoder_input_ids from labels (shifted right by 1)
    inputs['labels'] = targets['input_ids']
    
    return inputs

# Preprocess datasets
print("\nPreprocessing train dataset...")
ds_train = ds['train'].map(preprocess_function, batched=True, remove_columns=ds['train'].column_names)
print(f"Train: {len(ds_train)} samples preprocessed")

print("Preprocessing validation dataset...")
ds_val = ds['validation'].map(preprocess_function, batched=True, remove_columns=ds['validation'].column_names)
print(f"Validation: {len(ds_val)} samples preprocessed")

Tokenizer loaded: google/byt5-base
Vocab size: 256

Preprocessing train dataset...


Map:   0%|          | 0/2269 [00:00<?, ? examples/s]

Train: 2269 samples preprocessed
Preprocessing validation dataset...


Map:   0%|          | 0/252 [00:00<?, ? examples/s]

Validation: 252 samples preprocessed


## Load Model & Apply Quantization + LoRA

In [5]:
# Setup quantization config if needed
bnb_config = None
load_in_8bit = False

if CONFIG['quant_bits'] == 4:
    print("Setting up 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16
    )
elif CONFIG['quant_bits'] == 8:
    print("Setting up 8-bit quantization...")
    load_in_8bit = True

# Load model
print(f"\nLoading model: {CONFIG['model_name']}...")
if bnb_config is not None:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        CONFIG['model_name'],
        device_map='auto',
        quantization_config=bnb_config
    )
elif load_in_8bit:
    model = AutoModelForSeq2SeqLM.from_pretrained(
        CONFIG['model_name'],
        device_map='auto',
        load_in_8bit=True
    )
else:
    # Non-quantized: explicitly move to device
    model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['model_name'])
    model = model.to(DEVICE)

print(f"Model loaded on device: {next(model.parameters()).device}")

# Prepare for k-bit training if quantized
if CONFIG['quant_bits'] in (4, 8):
    print("Preparing model for k-bit training...")
    model = prepare_model_for_kbit_training(model)

# Apply LoRA
print("\nApplying LoRA...")
lora_config = LoraConfig(
    r=CONFIG['lora_r'],
    lora_alpha=CONFIG['lora_alpha'],
    target_modules=CONFIG['lora_targets'],
    lora_dropout=CONFIG['lora_dropout'],
    bias='none',
    task_type='SEQ_2_SEQ_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


Loading model: google/byt5-base...


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded on device: mps:0

Applying LoRA...
trainable params: 2,211,840 || all params: 583,865,088 || trainable%: 0.3788


## Training Loop

In [ ]:
# Setup training arguments
# Use bf16 instead of fp16 on Apple Silicon (fp16 not fully supported on MPS)
use_bf16 = DEVICE == 'mps'

# Create data collator for seq2seq - handles padding and masking correctly
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir=CONFIG['output_dir'],
    num_train_epochs=CONFIG['num_train_epochs'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation_steps'],
    per_device_train_batch_size=CONFIG['per_device_train_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_eval_batch_size'],
    learning_rate=CONFIG['learning_rate'],
    warmup_steps=CONFIG['warmup_steps'],
    weight_decay=CONFIG['weight_decay'],
    logging_steps=CONFIG['logging_steps'],
    eval_steps=CONFIG['eval_steps'],
    save_steps=CONFIG['save_steps'],
    save_total_limit=CONFIG['save_total_limit'],
    eval_strategy='steps',
    predict_with_generate=True,
    bf16=use_bf16,  # Use bf16 on MPS instead of fp16
    fp16=(not use_bf16),  # Use fp16 only on non-MPS
    gradient_checkpointing=False,
    remove_unused_columns=False,
)

print(f"Training with device={DEVICE}, bf16={use_bf16}, fp16={not use_bf16}\n")
print(f"Epochs: {CONFIG['num_train_epochs']}, Effective Batch Size: {CONFIG['gradient_accumulation_steps'] * CONFIG['per_device_train_batch_size']}\n")

# Create trainer - MUST pass tokenizer and data_collator for seq2seq
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    data_collator=data_collator,  # Proper seq2seq data collator
    tokenizer=tokenizer,  # Required for generation during evaluation
)

# Start training
print("Starting training...\n")
train_result = trainer.train()

Training with device=mps, bf16=True, fp16=False

Epochs: 3, Effective Batch Size: 64

Starting training...



/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss
108,1152.696250,14.935405


## Validation & Metrics

In [7]:
# Evaluate on validation set
print("Evaluating on validation set...")
eval_result = trainer.evaluate()

print("\n=== Training Results ===")
print(f"Training loss: {train_result.training_loss:.4f}")

print("\n=== Evaluation Results ===")
for key, val in eval_result.items():
    if isinstance(val, float):
        print(f"{key}: {val:.4f}")
    else:
        print(f"{key}: {val}")

Evaluating on validation set...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Step
1152.696250,14.935405,108



=== Training Results ===
Training loss: 1136.1890

=== Evaluation Results ===
eval_loss: 14.9354


## Save Validation Predictions

In [20]:
# Get validation predictions and metrics in one call
print("Getting validation predictions and metrics from trainer...")

# Single call - gets predictions, metrics, everything
predictions_output = trainer.predict(ds_val)
pred_ids = predictions_output.predictions
metrics = predictions_output.metrics

print("\n=== Training Results ===")
print(f"Training loss: {train_result.training_loss:.4f}")

print("\n=== Validation Metrics ===")
for key, val in metrics.items():
    if isinstance(val, float):
        print(f"{key}: {val:.4f}")
    else:
        print(f"{key}: {val}")

# Save validation predictions to CSV
SAVE_VALIDATION_PRED = True

if SAVE_VALIDATION_PRED:
    import os
    import csv
    print("\nDecoding and saving validation predictions to CSV...")
    
    # Batch decode all predictions at once (much faster and safer)
    decoded_preds = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    
    # Build output
    val_rows = []
    raw_val = ds['validation']
    
    for i, ex in enumerate(raw_val):
        input_text = ex.get('input_text', '')
        target_text = ex.get('target_text', '')
        pred_text = decoded_preds[i]
        
        val_rows.append({
            'input_text': input_text,
            'target_text': target_text,
            'predicted_translation': pred_text,
        })
        
        if (i + 1) % 100 == 0:
            print(f"  Processed {i+1}/{len(raw_val)} predictions")
    
    # Save
    out_path = os.path.join(CONFIG['output_dir'], 'validation_predictions.csv')
    pd.DataFrame(val_rows).to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    print(f"Saved {len(val_rows)} validation predictions to: {out_path}")

Getting validation predictions and metrics from trainer...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)



=== Training Results ===
Training loss: 1133.0428

=== Validation Metrics ===
test_loss: 14.8999
test_runtime: 231.0842
test_samples_per_second: 1.0910
test_steps_per_second: 1.0910

Decoding and saving validation predictions to CSV...
  Processed 100/252 predictions
  Processed 200/252 predictions
Saved 252 validation predictions to: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt/validation_predictions.csv


## Save Model

In [8]:
# Save the trained model and tokenizer
print(f"Saving model to {CONFIG['output_dir']}...")
trainer.save_model(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
print("Model saved!")

Saving model to /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt...
Model saved!


## Load Model from Checkpoint

In [9]:
from peft import PeftModel

# Load the base model
checkpoint_dir = CONFIG['output_dir']

# Load base model
base_model = AutoModelForSeq2SeqLM.from_pretrained(CONFIG['model_name'])
base_model = base_model.to(DEVICE)

# Load LoRA adapters from checkpoint
model = PeftModel.from_pretrained(base_model, checkpoint_dir)
model.eval()

print(f"Model loaded from: {checkpoint_dir}")
print(f"Model is on device: {next(model.parameters()).device}")


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model loaded from: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt
Model is on device: mps:0


## Load Test Data (Blind Test)

In [10]:
# Load test_clean.csv - only use transliteration_with_annotations (no translation available)
test_df = pd.read_csv(CONFIG['test_file'], dtype=str)
print(f"Test dataset: {len(test_df)} rows")
print(f"Columns: {list(test_df.columns)}")

# Extract input_text from transliteration_with_annotations
test_inputs = test_df['transliteration_with_annotations'].fillna('').tolist()
print(f"\nTest samples with input: {sum(1 for x in test_inputs if x.strip())}")

# Show sample
if test_inputs:
    for input_text in test_inputs:
        print(f"Sample test input (first 200 chars): {input_text[:200]}...")

Test dataset: 4 rows
Columns: ['id', 'text_id', 'line_start', 'line_end', 'transliteration', 'transliteration_clean', 'transliteration_with_annotations']

Test samples with input: 4
Sample test input (first 200 chars): um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il <gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim {(a small Ass. trading colony) OA;} qí-bi„-ma mup-pu-um aa a-lim{ki} i-li-kam {to go}...
Sample test input (first 200 chars): i-na mup-pì-im aa a-lim{ki} ia-tù u„-mì-im a-nim {this, those, this matter, one way or another, various, some ..., others ..., thus, in the same way} ma-ma-an {somebody, who(so)ever, nobody; who(so)ev...
Sample test input (first 200 chars): ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam {there, from there} lu a-na aí-mì-im a-na É.GAL-lim {palace service; palace, review p., arsenal, rear p., p. of rest, the government, palace overseer, queen, pal...
Sample test input (first 200 chars): me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-bar-ra-tim {(a small Ass. tradin

## Inference on Test Data (Blind Test)

In [15]:
# Load model for inference (already in memory, but reload if needed after restart)
model.eval()

print(f"Generating predictions for {len(test_inputs)} test samples...")
print(f"Using device: {DEVICE}\n")

# Batch inference - MUCH faster than per-sample
# Start with batch_size=16; reduce if OOM
BATCH_SIZE = 16
predictions = []

def generate_batch(batch_texts, batch_size=BATCH_SIZE):
    """Generate predictions for a batch of texts. Reduce batch_size on OOM."""
    global BATCH_SIZE
    try:
        # Tokenize batch
        inputs = tokenizer(
            batch_texts,
            truncation=True,
            padding='max_length',
            max_length=CONFIG['max_input_length'],
            return_tensors='pt'
        )
        
        # Move to device
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        
        # Generate batch
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_length=CONFIG['max_target_length'],
                num_beams=1,
                do_sample=False
            )
        
        # Batch decode (much faster than decoding one-by-one)
        batch_preds = tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        return batch_preds
        
    except torch.cuda.OutOfMemoryError:
        print(f"⚠ OOM at batch_size={batch_size}. Reducing to {batch_size//2}...")
        BATCH_SIZE = batch_size // 2
        # Recursively try smaller batch
        return generate_batch(batch_texts, batch_size // 2)

# Process in batches
print(f"Using batch size: {BATCH_SIZE}")
start_time = torch.cuda.Event(enable_timing=True) if DEVICE == 'cuda' else None
if start_time:
    start_time.record()

for batch_start in range(0, len(test_inputs), BATCH_SIZE):
    batch_end = min(batch_start + BATCH_SIZE, len(test_inputs))
    batch_texts = test_inputs[batch_start:batch_end]
    
    # Filter out empty inputs
    batch_preds = []
    for text in batch_texts:
        if text and text.strip():
            batch_preds.append(text)
        else:
            batch_preds.append('')  # placeholder
    
    # Only generate for non-empty
    non_empty_indices = [i for i, t in enumerate(batch_texts) if t and t.strip()]
    if non_empty_indices:
        non_empty_texts = [batch_texts[i] for i in non_empty_indices]
        non_empty_preds = generate_batch(non_empty_texts)
        
        # Map back to full batch
        for idx, pred_idx in enumerate(non_empty_indices):
            batch_preds[pred_idx] = non_empty_preds[idx]
    
    predictions.extend(batch_preds)
    print(f"  Generated {batch_end}/{len(test_inputs)} predictions")

if start_time:
    start_time.record()
    print(f"\n✓ Batch inference completed in {start_time.elapsed_time() / 1000:.2f}s")
else:
    print(f"\n✓ Batch inference completed.")

print(f"Generated {len(predictions)} predictions.")

# Show all sample predictions
num_samples_to_show = min(5, len(predictions))  # Show up to 5 samples
for i in range(num_samples_to_show):
    print(f"\nSample {i+1}:")
    print(f"  Input: {test_inputs[i][:150]}...")
    print(f"  Prediction: {predictions[i][:150]}...")

Generating predictions for 4 test samples...
Using device: mps

Using batch size: 16
  Generated 4/4 predictions

✓ Batch inference completed.
Generated 4 predictions.

Sample 1:
  Input: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il <gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim {(a small Ass. trading colony) OA;} qí-bi„-ma mup-pu-...
  Prediction: a aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-r...

Sample 2:
  Input: i-na mup-pì-im aa a-lim{ki} ia-tù u„-mì-im a-nim {this, those, this matter, one way or another, various, some ..., others ..., thus, in the same way} ...
  Prediction: -mu-ni i-na mup-pì-im aa a-lim {to suffer} ia-tù u„-mu-ni i-na mup-pì-im aa a-lim {to suffer} ia-tù u„-mu-ni i-na mup-pì-im aa a-lim {to suffer} ia-tù...

Sample 3:
  Input: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam {there, from there} lu a-na aí-mì-im a-na É.GAL-lim {palace service; palace, review p., a

## Save Predictions to CSV

In [12]:
# Create output dataframe
output_df = pd.DataFrame({
    'transliteration_with_annotations': test_inputs,
    'predicted_translation': predictions
})

# Save to CSV
output_path = CONFIG['predictions_file']
output_df.to_csv(output_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Predictions saved to: {output_path}")
print(f"Total rows: {len(output_df)}")

# Show summary
print(f"\nOutput CSV summary:")
print(output_df.head())
print(f"\nPredictions with non-empty output: {(output_df['predicted_translation'].str.len() > 0).sum()}")

Predictions saved to: /Users/adityabehera/Desktop/GATECH/Spring 2026/NLP/deep-past-akkadian/byt5_lora_ckpt/test_predictions.csv
Total rows: 4

Output CSV summary:
                    transliteration_with_annotations  \
0  um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il <gap>...   
1  i-na mup-pì-im aa a-lim{ki} ia-tù u„-mì-im a-n...   
2  ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam {there,...   
3  me-+e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-ba...   

                               predicted_translation  
0  a aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa-ra aa...  
1  -mu-ni i-na mup-pì-im aa a-lim {to suffer} ia-...  
2   an ai-mì-im a-na É.GAL-lim {palace service; p...  
3  -ra-tim {to raise a slave; to raise a slave; t...  

Predictions with non-empty output: 4
